In [1]:
%%capture
!pip install torch torchvision torchaudio tqdm pandas Pillow

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import os
from torchvision import transforms
from tqdm import tqdm

import torch.utils.data as data
from torch.utils.data import Dataset, Subset, random_split
from PIL import Image
from torch.utils.data import DataLoader

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!cp -r "/content/drive/MyDrive/Datasets/Brain Tumor Dataset" /content/local_dataset

In [5]:
DATASET_PATH = "/content/local_dataset"

In [6]:
class BrainTumorDataset(Dataset):
  CLASSES = ["glioma", "meningioma", "notumor", "pituitary"] # names of the folders are the classes

  def __init__(self, root_dir, transform=None):
    self.root_dir = root_dir
    self.transform = transform

    # one-hot index mapping:
    self.class_to_idx = {cls: idx for idx, cls in enumerate(self.CLASSES)}

    # collect all (image, path) pairs:
    self.samples = self._load_samples()


  def _load_samples(self):
    """walk `root_dir` and collect all (image, path) pairs"""
    samples = []
    valid_extensions = {".jpg", ".jpeg", ".png", ".bmp"} 

    for class_name in self.CLASSES:
      class_dir = os.path.join(self.root_dir, class_name)

      if not os.path.isdir(class_dir):
        raise FileNotFoundError(f"Couldn't find directory {class_dir}")

      for file_name in os.listdir(class_dir):
        ext = os.path.splitext(file_name)[1].lower()
        if ext in valid_extensions:
          img_path = os.path.join(class_dir, file_name)
          samples.append((img_path, class_name))

    return samples


  def _one_hot(self, class_name):
    """returns a one-hot encoded tensor for the given `class_name`"""

    idx = self.class_to_idx[class_name]
    label = torch.zeros(len(self.CLASSES))
    label[idx] = 1.0
    return label


  def __len__(self):
    return len(self.samples)


  def __getitem__(self, idx):
    img_path, class_name = self.samples[idx]
    image = Image.open(img_path).convert("L")

    if self.transform:
      image = self.transform(image)

    label = self._one_hot(class_name)
    return image, label

In [7]:
MEAN = 0.1856 # calculated those values across dataset
STD = 0.1892

training_path = os.path.join(DATASET_PATH, "Training")
testing_path = os.path.join(DATASET_PATH, "Testing")

transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1), # this is very reasonable for MRI scans to reduce computation and noise
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD), # better not to use ImageNet mean and std in such case
    ])

train_dataset = BrainTumorDataset(root_dir=training_path, transform=transform)
test_dataset = BrainTumorDataset(root_dir=testing_path, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# SANITY CHECK
image, label = train_dataset[-1]
print(f"Image shape: {image.shape}")
print(f"Corresponding Label: {label}")

Image shape: torch.Size([1, 224, 224])
Corresponding Label: tensor([0., 0., 0., 1.])


In [8]:
class BrainTumorClassifierV2(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        def conv_block(in_ch, out_ch):
            # Conv -> BN -> ReLU -> Pool
            # no dropout inside conv blocks
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2),
            )

        self.features = nn.Sequential(
            conv_block(1, 32),   # 224 -> 112
            conv_block(32, 64),  # 112 -> 56
            conv_block(64, 128), # 56  -> 28
            conv_block(128, 256),# 28  -> 14
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),  # 256 × 4 × 4 = 4096
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),               # dropout here before final layer
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [9]:

# ==============================================
#  HYPERPARAMETERS
# ==============================================
BATCH_SIZE   = 32
NUM_CLASSES  = 4
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Two-phase epochs
PHASE1_EPOCHS = 10   # backbone frozen,  head-only training
PHASE2_EPOCHS = 30   # full fine-tuning  (backbone + head)

# Learning rates
PHASE1_LR     = 1e-3
PHASE2_LR     = 1e-4
WEIGHT_DECAY  = 1e-4

# Pre-computed dataset statistics 
MEAN = 0.1856
STD  = 0.1892

print(f"\033[94mUsing device: {DEVICE}\033[0m")


# ==============================================
#  TRANSFORMS
# ==============================================
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),              # Random crop beats plain resize
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),               # Small random shifts
        shear=5
    ),
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3                         # Works on grayscale too
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[MEAN], std=[STD]),
    transforms.RandomErasing(p=0.2),        # Randomly mask patches -> robust features
])

test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[MEAN], std=[STD]),
])


# ==============================================
#  DATASET & DATALOADER
# ==============================================
train_dataset = BrainTumorDataset(
    root_dir=os.path.join(DATASET_PATH, "Training"),
    transform=train_transform
)
test_dataset = BrainTumorDataset(
    root_dir=os.path.join(DATASET_PATH, "Testing"),
    transform=test_transform
)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,      # Parallel loading - speeds up GPU utilization
    pin_memory=True     # Faster CPU -> GPU transfer
)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print(
    f"\033[92mTraining samples: {len(train_dataset)} | "
    f"Testing samples: {len(test_dataset)}\033[0m"
)

Using device: cuda
Training samples: 5727 | Testing samples: 1311


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [10]:
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights


# ==============================================
#  MODEL  -  EfficientNet-B3 (pretrained)
# ==============================================
class BrainTumorEfficientNet(nn.Module):
    """
    EfficientNet-B3 backbone fine-tuned for grayscale brain-MRI classification.

    Key adaptations vs a raw EfficientNet:
      * First conv layer replaced to accept 1-channel (grayscale) input.
        The pretrained RGB weights are averaged across channels so no
        information is thrown away.
      * The original classifier head is replaced with Dropout → Linear(4)
        so we control regularisation precisely.
    """

    def __init__(self, num_classes: int = 4):
        super().__init__()

        # -- Load pretrained backbone ------------------------------
        backbone = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)

        # -- Adapt first conv: RGB (3-ch) -> Grayscale (1-ch) --------
        old_conv = backbone.features[0][0]   # Conv2d(3, 40, ...)
        new_conv = nn.Conv2d(
            in_channels=1,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False,
        )
        # Average the 3 RGB weight planes into 1 -> preserves pretrained knowledge
        new_conv.weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)
        backbone.features[0][0] = new_conv

        self.backbone = backbone

        # --- Replace classifier head -----------------------------
        in_features = self.backbone.classifier[1].in_features  # 1536 for B3
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.4),                    # Dropout BEFORE final linear
            nn.Linear(in_features, num_classes),  # Raw logits out
        )

    def freeze_backbone(self):
        """Freeze everything except the classifier head (Phase 1)."""
        for param in self.backbone.features.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        """Unfreeze the full network for fine-tuning (Phase 2)."""
        for param in self.backbone.features.parameters():
            param.requires_grad = True

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)


In [11]:
# ==============================================
#  LOSS
# ==============================================
# label_smoothing=0.1 prevents the model becoming overconfident
# and improves generalisation on small datasets
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

In [12]:
# ==============================================
#  TRAINING & EVALUATION LOOPS
# ==============================================
def train_one_epoch(model, loader, optimizer, epoch, total_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    # Progress Bar
    pbar = tqdm(
        enumerate(loader),
        total=len(loader),
        desc=f"Epoch {epoch+1}/{total_epochs}"
    )
    for batch_idx, (images, labels) in pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels.argmax(dim=1))
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        correct += (outputs.argmax(dim=1) == labels.argmax(dim=1)).sum().item()
        total += labels.size(0)

        pbar.set_postfix({"loss": f"{running_loss/(batch_idx+1):.4f}"})

    return running_loss / len(loader), 100 * correct / total


def evaluate(model, loader):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels.argmax(dim=1))

            running_loss += loss.item()
            correct += (outputs.argmax(dim=1) == labels.argmax(dim=1)).sum().item()
            total += labels.size(0)

    return running_loss / len(loader), 100 * correct / total

In [13]:
# ==============================================
#  TRAINING RUNNER
# ==============================================
def run_phase(model, loader_train, loader_test,
              lr, epochs, phase_name, start_epoch=0):
    """
    Generic training phase.
    Returns (train_losses, train_accs, test_losses, test_accs, best_acc).
    """
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )
    # Cosine annealing smoothly decays LR -> much better than a flat rate
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    best_acc = 0.0
    total_epochs = start_epoch + epochs

    for i in range(epochs):
        epoch = start_epoch + i
        tl, ta = train_one_epoch(model, loader_train, optimizer, epoch, total_epochs)
        vl, va = evaluate(model, loader_test)
        scheduler.step()

        history["train_loss"].append(tl)
        history["train_acc"].append(ta)
        history["test_loss"].append(vl)
        history["test_acc"].append(va)

        current_lr = optimizer.param_groups[0]["lr"]
        print(
            f"\n[{phase_name}] Epoch {epoch+1}/{total_epochs} | "
            f"LR: {current_lr:.6f}\n"
            f"  Train  → Loss: {tl:.4f} | Acc: {ta:.2f}%\n"
            f"  Test   → Loss: {vl:.4f} | Acc: {va:.2f}%"
        )

        if va > best_acc:
            best_acc = va
            torch.save(model.state_dict(), f"best_model_{phase_name.lower().replace(' ','_')}.pth")
            print(f"\033[92m  ✓ Saved new best ({phase_name}): {best_acc:.2f}%\033[0m")

    return history, best_acc

In [14]:
# ==============================================
#  MAIN
# ==============================================
model = BrainTumorEfficientNet(num_classes=NUM_CLASSES).to(DEVICE)

# -- Phase 1: train head only ---------------------------------------
print("\n\033[94m══ PHASE 1: Head-only training (backbone frozen) ══\033[0m")
model.freeze_backbone()

hist1, best1 = run_phase(
    model, train_loader, test_loader,
    lr=PHASE1_LR,
    epochs=PHASE1_EPOCHS,
    phase_name="Phase 1",
    start_epoch=0,
)

# -- Phase 2: full fine-tuning ---------------------------------------
print("\n\033[94m══ PHASE 2: Full fine-tuning (all layers) ══\033[0m")
model.unfreeze_backbone()

# Load the best Phase 1 weights before unlocking everything
model.load_state_dict(torch.load("best_model_phase_1.pth", map_location=DEVICE))

hist2, best2 = run_phase(
    model, train_loader, test_loader,
    lr=PHASE2_LR,           # 10× smaller LR to avoid destroying pretrained weights
    epochs=PHASE2_EPOCHS,
    phase_name="Phase 2",
    start_epoch=PHASE1_EPOCHS,
)

# -- Final summary ---------------------------------------
overall_best = max(best1, best2)
print(
    f"\n\033[92m══ Training complete ══\n"
    f"   Phase 1 best test acc : {best1:.2f}%\n"
    f"   Phase 2 best test acc : {best2:.2f}%\n"
    f"   Overall best          : {overall_best:.2f}%\033[0m"
)

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 183MB/s]



══ PHASE 1: Head-only training (backbone frozen) ══


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Evaluating: 100%|██████████| 41/41 [00:03<00:00, 11.59it/s]



[Phase 1] Epoch 1/10 | LR: 0.000976
  Train  → Loss: 0.9037 | Acc: 72.76%
  Test   → Loss: 0.8189 | Acc: 76.66%
  ✓ Saved new best (Phase 1): 76.66%


...


[Phase 2] Epoch 38/40 | LR: 0.000001
  Train  → Loss: 0.3692 | Acc: 99.49%
  Test   → Loss: 0.3650 | Acc: 99.24%



Evaluating: 100%|██████████| 41/41 [00:04<00:00,  9.23it/s]


[Phase 2] Epoch 39/40 | LR: 0.000000
  Train  → Loss: 0.3721 | Acc: 99.28%
  Test   → Loss: 0.3643 | Acc: 99.31%



Evaluating: 100%|██████████| 41/41 [00:03<00:00, 10.38it/s]


[Phase 2] Epoch 40/40 | LR: 0.000000
  Train  → Loss: 0.3686 | Acc: 99.46%
  Test   → Loss: 0.3626 | Acc: 99.31%

══ Training complete ══
   Phase 1 best test acc : 81.62%
   Phase 2 best test acc : 99.39%
   Overall best          : 99.39%


In [16]:
import json

# Save model weights
torch.save(model.state_dict(), "best_model.pth")

# Save class labels (order must match your dataset's class_to_idx)
class_names = train_dataset.CLASSES  # ['glioma', 'meningioma', 'notumor', 'pituitary']
with open("class_names.json", "w") as f:
    json.dump(class_names, f)

# Download both files to your machine from Colab
from google.colab import files
files.download("best_model.pth")
files.download("class_names.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>